Based on our anomaly detection findings, the Eurostat datasets mix absolute figures with percentages. Next steps:
1. Convert all percentage-based values into absolute numeric figures.
2. Fill remaining null values

In [0]:
import importlib
from config.config import *

from delta.tables import DeltaTable
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
for table, metric in [
    ("estat_tour_occ_arm", "tourism_value"),
    ("estat_tour_occ_nim", "nights_spent")
]:
    df = spark.read.format("delta").load(
        f"{silver_path}/{table}"
    )

    print(f"\n{'=' * 70}")
    print(table)
    print(f"{'=' * 70}")

    df.groupBy(
        "geo_code",
        "frequency",
        "residence_category",
        "activity_code",
        "unit_code"
    ).agg(
        count("*").alias("total_periods"),
        count(metric).alias("available_values"),
        min("period").alias("first_period"),
        max("period").alias("last_period")
    ).orderBy(
        "geo_code",
        "unit_code"
    ).show(30, truncate=False)

In [0]:
for table, metric in [
    ("estat_tour_occ_arm", "tourism_value"),
    ("estat_tour_occ_nim", "nights_spent")
]:

    df = spark.read.format("delta").load(
        f"{silver_path}/{table}"
    )

    print(f"\n{'=' * 60}")
    print(table)
    print(f"{'=' * 60}")

    df.filter(
        (col("unit_code") == "NR") &
        year(col("period")).between(2010, 2024)
    ).groupBy(
        year(col("period")).alias("year")
    ).agg(
        count("*").alias("total"),
        count(metric).alias("observed"),
        count(when(col(metric).isNull(), 1)).alias("missing")
    ).withColumn(
        "completeness_pct",
        round(col("observed") / col("total") * 100, 2)
    ).orderBy("year").show(20)

In [0]:
for table in ["estat_tour_occ_arm", "estat_tour_occ_nim"]:

    df = spark.read.format("delta").load(
        f"{silver_path}/{table}"
    )

    print(f"\n{'='*70}")
    print(table)
    print(f"{'='*70}")

    df.groupBy("unit_code").agg(
        count("*").alias("records"),
        count("tourism_value" if table == "estat_tour_occ_arm" else "nights_spent").alias("available_values"),
        countDistinct("country_iso3").alias("countries"),
        min("period").alias("first_period"),
        max("period").alias("last_period")
    ).orderBy("unit_code").show(30, truncate=False)